# Proyecto BI — Ventas de Retail (Superstore Sales)
**Institución Universitaria ITM · Análisis de Datos · 2026-1**

**Dataset:** Superstore Sales (`train.csv`) — 9 800 registros de órdenes de venta
al por menor en Estados Unidos (2015-2018), con variables de cliente, producto,
envío y ventas.

In [ ]:
# ─── Importaciones globales ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)

# Carga del dataset
df = pd.read_csv("train.csv", encoding="utf-8")

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
print(df.dtypes)

---
## Fase 1 · Comprensión del Problema

### 1.1 Descripción del contexto del dataset

El dataset **Superstore Sales** contiene registros de órdenes realizadas en una
cadena minorista estadounidense entre 2015 y 2018. Cada fila representa una línea
de pedido e incluye información sobre:

| Dimensión | Variables |
|-----------|-----------|
| Tiempo | `Order Date`, `Ship Date` |
| Cliente | `Customer ID`, `Customer Name`, `Segment` |
| Geografía | `Country`, `City`, `State`, `Postal Code`, `Region` |
| Logística | `Ship Mode` |
| Producto | `Product ID`, `Category`, `Sub-Category`, `Product Name` |
| Finanzas | `Sales` |

### 1.2 Identificación del problema de negocio

La empresa necesita entender **qué factores impulsan o frenan las ventas** para
optimizar la asignación de recursos comerciales, logísticos y de marketing.
En concreto, se desconoce si existen diferencias significativas entre regiones,
segmentos de clientes y modalidades de envío que justifiquen estrategias
diferenciadas.

### 1.3 Preguntas clave

1. **¿Qué región, categoría y segmento de clientes generan más ventas?**
2. **¿El modo de envío influye en el valor promedio de la venta?**
3. **¿Existen diferencias estadísticamente significativas entre las ventas
   medianas de los distintos segmentos de clientes (Consumer, Corporate, Home Office)?**
4. **¿Qué productos o sub-categorías concentran el mayor volumen de ingresos?**
5. **¿Cómo evolucionan las ventas a lo largo del tiempo?**

In [ ]:
# ─── Fase 1 · Exploración inicial ─────────────────────────────────────────────

print("=" * 60)
print("FASE 1 — Exploración inicial del dataset")
print("=" * 60)

print("\n── Primeras filas ──")
print(df.head(3).to_string())

print("\n── Estadísticas descriptivas de Sales ──")
print(df["Sales"].describe().round(2))
print(f"\n  Skewness: {df['Sales'].skew():.2f}  (>1 = sesgo alto, distribución no normal)")
print(f"  Media:    ${df['Sales'].mean():.2f}")
print(f"  Mediana:  ${df['Sales'].median():.2f}  → ratio media/mediana = {df['Sales'].mean()/df['Sales'].median():.1f}x")

print("\n── Distribución por Segmento ──")
print(df["Segment"].value_counts())

print("\n── Distribución por Región ──")
print(df["Region"].value_counts())

print("\n── Distribución por Categoría ──")
print(df["Category"].value_counts())

print("\n── Distribución por Ship Mode ──")
print(df["Ship Mode"].value_counts())

### Decisión metodológica — Transformación log(Sales + 1)

Al explorar la variable `Sales` encontramos los siguientes valores:

| Estadístico | Valor |
|-------------|-------|
| Media | $230.77 |
| Mediana | $54.49 |
| Ratio media/mediana | 4.2× |
| Skewness | **12.98** |
| Máximo | $22,638.48 |

Un **skewness de 12.98** es extremadamente alto (valores >1 ya se consideran
sesgados; >3 son severos). Esto ocurre porque el 4.7 % de los pedidos
(aquellos con `Sales > $1,000`) concentran el **43.2 % de los ingresos totales**:
pocas ventas de gran valor "estiran" la cola derecha de la distribución.

**¿Por qué log(Sales + 1) y no otra transformación?**

- **`log(x + 1)`** (logaritmo natural con corrección de +1 para evitar log(0)):
  comprime la cola larga de forma proporcional, convirtiendo diferencias
  multiplicativas en diferencias aditivas. Es la transformación estándar para
  variables monetarias con sesgo positivo.
- Alternativas como raíz cuadrada o Box-Cox también reducen el sesgo, pero
  `log1p` es más interpretable: un incremento de 1 unidad en la escala log
  equivale aproximadamente a un incremento porcentual constante en Sales.
- **Resultado:** el skewness pasa de 12.98 → **0.28**, obteniendo una
  distribución prácticamente normal (campana simétrica visible en el histograma).

> **Nota importante:** la transformación se aplica **solo para visualización**.
> Los KPIs y agregados de negocio (sumas, medianas) se calculan siempre sobre
> los valores originales de `Sales`, que son los dólares reales.

In [ ]:
# ─── Fase 1 · Visualización exploratoria ──────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Fase 1 — Distribución general del dataset", fontsize=14, fontweight="bold")

# Ventas por Segmento
seg_sales = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
axes[0, 0].bar(seg_sales.index, seg_sales.values, color=sns.color_palette("Set2", 3))
axes[0, 0].set_title("Ventas totales por Segmento")
axes[0, 0].set_ylabel("Ventas (USD)")
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Ventas por Región
reg_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
axes[0, 1].bar(reg_sales.index, reg_sales.values, color=sns.color_palette("Set2", 4))
axes[0, 1].set_title("Ventas totales por Región")
axes[0, 1].set_ylabel("Ventas (USD)")
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Ventas por Categoría
cat_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
axes[1, 0].bar(cat_sales.index, cat_sales.values, color=sns.color_palette("Set2", 3))
axes[1, 0].set_title("Ventas totales por Categoría")
axes[1, 0].set_ylabel("Ventas (USD)")
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

log_sales = np.log1p(df["Sales"])
axes[1, 1].hist(log_sales, bins=50, color="#4C72B0", edgecolor="white")
axes[1, 1].set_title(
    f"Distribución de log(Sales+1)\n"
    f"Skewness original={df['Sales'].skew():.1f} → log={log_sales.skew():.2f}"
)
axes[1, 1].set_xlabel("log(Ventas + 1)")
axes[1, 1].set_ylabel("Frecuencia")
# Ticks en escala original para referencia
tick_vals = [0, 10, 50, 100, 500, 1000, 5000, 22000]
axes[1, 1].set_xticks([np.log1p(v) for v in tick_vals])
axes[1, 1].set_xticklabels([f"${v:,.0f}" for v in tick_vals], rotation=35, ha="right", fontsize=7)

plt.tight_layout()
plt.savefig("fase1_exploracion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fase1_exploracion.png")